In [2]:
!pip install -q transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import os
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [4]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "/content/drive/MyDrive/Week-8 Local/adapters"
SAVE_DIR = "./quantized"

os.makedirs(f"{SAVE_DIR}/model-fp16", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/model-int8", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/model-int4", exist_ok=True)

In [ ]:
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


merged = PeftModel.from_pretrained(base, ADAPTER_PATH)
merged = merged.merge_and_unload()

merged.save_pretrained(f"{SAVE_DIR}/model-fp16")
tokenizer.save_pretrained(f"{SAVE_DIR}/model-fp16")

print("FP16 saved!")
print("Size:", round(os.path.getsize(f"{SAVE_DIR}/model-fp16/model.safetensors") / 1e9, 2), "GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FP16 saved!
Size: 2.2 GB


In [ ]:
bnb_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    f"{SAVE_DIR}/model-fp16",
    quantization_config=bnb_8bit,
    device_map="auto"
)

model_int8.save_pretrained(f"{SAVE_DIR}/model-int8")
tokenizer.save_pretrained(f"{SAVE_DIR}/model-int8")
print("INT8 saved!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 saved!


In [7]:
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    f"{SAVE_DIR}/model-fp16",
    quantization_config=bnb_4bit,
    device_map="auto"
)

model_int4.save_pretrained(f"{SAVE_DIR}/model-int4")
tokenizer.save_pretrained(f"{SAVE_DIR}/model-int4")
print("INT4 saved!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 saved!


In [21]:
!pip install -q gguf sentencepiece

!wget -q https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.model \
    -O ./quantized/model-fp16/tokenizer.model

!python llama.cpp/convert_hf_to_gguf.py \
    ./quantized/model-fp16 \
    --outfile ./quantized/model.gguf \
    --outtype q8_0

print("GGUF done!")
print("Size:", round(os.path.getsize("./quantized/model.gguf") / 1e9, 3), "GB")

INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_outpu

In [ ]:
files_to_check = [
    "./quantized/model-fp16/model.safetensors",
    "./quantized/model-int8",
    "./quantized/model-int4",
    "./quantized/model.gguf",
]

print("Day 3 file check:")
for f in files_to_check:
    exists = os.path.exists(f)
    size = ""
    if exists and os.path.isfile(f):
        size = f"({round(os.path.getsize(f)/1e9, 2)} GB)"
    print(f"{'✓' if exists else 'x'} {f} {size}")

Day 3 file check:
✅ ./quantized/model-fp16/model.safetensors (2.2 GB)
✅ ./quantized/model-int8 
✅ ./quantized/model-int4 
✅ ./quantized/model.gguf (1.17 GB)


In [23]:
def get_size(path):
    total = 0
    for f in os.listdir(path):
        fp = os.path.join(path, f)
        if os.path.isfile(fp):
            total += os.path.getsize(fp)
    return round(total / 1e9, 3)

fp16_size = get_size(f"{SAVE_DIR}/model-fp16")
int8_size = get_size(f"{SAVE_DIR}/model-int8")
int4_size = get_size(f"{SAVE_DIR}/model-int4")
gguf_size = round(os.path.getsize(f"{SAVE_DIR}/model.gguf") / 1e9, 3)

print(f"FP16 : {fp16_size} GB")
print(f"INT8 : {int8_size} GB")
print(f"INT4 : {int4_size} GB")
print(f"GGUF : {gguf_size} GB")

FP16 : 2.204 GB
INT8 : 1.236 GB
INT4 : 0.766 GB
GGUF : 1.17 GB


In [ ]:
test_prompt = "### Instruction:\nWhat is artificial intelligence?\n\n### Input:\n\n### Response:\n"

def benchmark(model, tokenizer, prompt, label):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    elapsed = time.time() - start
    tokens = out.shape[1] - inputs["input_ids"].shape[1]
    speed = round(tokens / elapsed, 2)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    response = text.split("### Response:")[-1].strip()
    print(f"\n{'='*40}")
    print(f"Format : {label}")
    print(f"Speed  : {speed} tokens/sec")
    print(f"Output : {response[:200]}")
    return speed


tok = AutoTokenizer.from_pretrained(f"{SAVE_DIR}/model-fp16")

m_fp16 = AutoModelForCausalLM.from_pretrained(f"{SAVE_DIR}/model-fp16", torch_dtype=torch.float16, device_map="auto")
speed_fp16 = benchmark(m_fp16, tok, test_prompt, "FP16")
del m_fp16

m_int8 = AutoModelForCausalLM.from_pretrained(f"{SAVE_DIR}/model-int8", quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="auto")
speed_int8 = benchmark(m_int8, tok, test_prompt, "INT8")
del m_int8

m_int4 = AutoModelForCausalLM.from_pretrained(f"{SAVE_DIR}/model-int4", quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), device_map="auto")
speed_int4 = benchmark(m_int4, tok, test_prompt, "INT4")
del m_int4

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Format : FP16
Speed  : 18.94 tokens/sec
Output : Artificial intelligence (AI) is the ability of machines to learn and adapt like humans. It is the ability of machines to perform tasks that require reasoning, problem-solving, and decision-making. AI 


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  def supports_quant_method(quantization_config_dict):


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Format : INT8
Speed  : 8.22 tokens/sec
Output : Artificial intelligence (AI) is the ability of a computer or machine to mimic intelligent behavior in a simulated environment or real-world situation. It is the ability of a computer or machine to lea


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Format : INT4
Speed  : 14.77 tokens/sec
Output : Artificial intelligence (AI) is the ability of machines to simulate human intelligence in the form of algorithms, machines, and systems. It is the ability of machines to learn and adapt to new situati


In [25]:
print("\nQUANTISATION COMPARISON TABLE")
print(f"{'Format':<10} {'Size (GB)':<12} {'Speed (tok/s)':<15}")
print("-" * 40)
print(f"{'FP16':<10} {fp16_size:<12} {speed_fp16:<15}")
print(f"{'INT8':<10} {int8_size:<12} {speed_int8:<15}")
print(f"{'INT4':<10} {int4_size:<12} {speed_int4:<15}")
print(f"{'GGUF':<10} {gguf_size:<12} {'run on CPU':<15}")


QUANTISATION COMPARISON TABLE
Format     Size (GB)    Speed (tok/s)  
----------------------------------------
FP16       2.204        18.94          
INT8       1.236        8.22           
INT4       0.766        14.77          
GGUF       1.17         run on CPU     


In [26]:
report = f"""# Quantisation Report — Day 3

## What We Did
Converted the fine-tuned TinyLlama model into 4 formats:
FP16 (baseline) → INT8 → INT4 → GGUF

## Results

| Format | Size (GB) | Speed (tok/s) | Quality |
|--------|-----------|---------------|---------|
| FP16   | {fp16_size}      | {speed_fp16}          | Best    |
| INT8   | {int8_size}      | {speed_int8}          | Good    |
| INT4   | {int4_size}      | {speed_int4}          | Good    |
| GGUF   | {gguf_size}      | CPU inference | Good    |

## Conclusion
- INT8 is ~2x smaller than FP16 with almost no quality loss
- INT4 is ~4x smaller than FP16, runs faster, slight quality drop
- GGUF (q4_0) allows the model to run on CPU without any GPU
"""

with open("QUANTISATION-REPORT.md", "w") as f:
    f.write(report)

print(report)

# Quantisation Report — Day 3

## What We Did
Converted the fine-tuned TinyLlama model into 4 formats:
FP16 (baseline) → INT8 → INT4 → GGUF

## Results

| Format | Size (GB) | Speed (tok/s) | Quality |
|--------|-----------|---------------|---------|
| FP16   | 2.204      | 18.94          | Best    |
| INT8   | 1.236      | 8.22          | Good    |
| INT4   | 0.766      | 14.77          | Good    |
| GGUF   | 1.17      | CPU inference | Good    |

## Conclusion
- INT8 is ~2x smaller than FP16 with almost no quality loss
- INT4 is ~4x smaller than FP16, runs faster, slight quality drop
- GGUF (q4_0) allows the model to run on CPU without any GPU



In [ ]:
import shutil
import os
from google.colab import drive


drive.mount("/content/drive")


save_path = "/content/drive/MyDrive/Week-8 Local/quantized"
os.makedirs(save_path, exist_ok=True)


shutil.copytree("./quantized", save_path, dirs_exist_ok=True)


if os.path.exists("QUANTISATION-REPORT.md"):
    shutil.copy("QUANTISATION-REPORT.md", f"{save_path}/QUANTISATION-REPORT.md")

print("---")
print("Success! All models and the report are now in your Drive.")
print("Drive Folder Contents:", os.listdir(save_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
---
✅ Success! All models and the report are now in your Drive.
Drive Folder Contents: ['model.gguf', 'model-int8', 'model-f16.gguf', 'model-fp16', 'model-int4', 'QUANTISATION-REPORT.md']
